# Centralized MountainCar Multi-Method Benchmark
Run BO, DR, BO+DR, DORAEMON, and BO+DORAEMON across selected seeds with optional warm-start checkpoint.

In [ ]:
from pathlib import Path
import importlib
import sys

cwd = Path.cwd().resolve()
root = cwd
if (cwd / 'TinySim').exists():
    root = cwd
elif (cwd.name == 'TinySim') and (cwd / 'mountaincar_methods.py').exists():
    root = cwd.parent

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

mountaincar_methods = None
try:
    import TinySim.mountaincar_methods as mountaincar_methods
except ModuleNotFoundError:
    if (cwd / 'mountaincar_methods.py').exists():
        if str(cwd) not in sys.path:
            sys.path.insert(0, str(cwd))
        import mountaincar_methods
    else:
        raise

mountaincar_methods = importlib.reload(mountaincar_methods)

EPISODE_STEPS = mountaincar_methods.EPISODE_STEPS
METHOD_ORDER = mountaincar_methods.METHOD_ORDER
PROFILE_PRESETS = mountaincar_methods.PROFILE_PRESETS
run_all_methods = mountaincar_methods.run_all_methods

print('CWD:', cwd)
print('Import module:', mountaincar_methods.__name__)
print('Available methods:', METHOD_ORDER)
print('Profiles:', list(PROFILE_PRESETS['BO'].keys()))
print('Episode steps (fixed for fairness):', EPISODE_STEPS)


In [ ]:
# Config
SEEDS = [42, 123]
METHODS = METHOD_ORDER.copy()  # ['BO', 'DR', 'BO+DR', 'DORAEMON', 'BO+DORAEMON']
RUN_PROFILE = 'smoke'  # smoke | prototype | edge
# Episode horizon is globally fixed for fairness in mountaincar_methods.py: EPISODE_STEPS = 400

# Set to None or '' to train from scratch (no warm-start checkpoint)
# CHECKPOINT_PATH = None
CHECKPOINT_PATH = None

# Optional: force device used by method notebooks ('cpu' or 'cuda').
# Leave None to keep each notebook's default selection.
DEVICE = None

OUTPUT_ROOT = Path('TinySim/runs')

print('SEEDS:', SEEDS)
print('METHODS:', METHODS)
print('RUN_PROFILE:', RUN_PROFILE)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('DEVICE:', DEVICE)

In [ ]:
benchmark = run_all_methods(
    seeds=SEEDS,
    methods=METHODS,
    profile=RUN_PROFILE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    output_root=OUTPUT_ROOT,
)

master_df = benchmark['master_df']
aggregate_df = benchmark['aggregate_df']

print('Run dir:', benchmark['run_dir'])
print('Master CSV:', benchmark['master_csv'])
print('Master JSON:', benchmark['master_json'])
print('Aggregate CSV:', benchmark['aggregate_csv'])
print('Plots:', benchmark['plot_paths'])

In [ ]:
master_df

In [ ]:
aggregate_df

In [ ]:
# Quick checks
expected_rows = len(SEEDS) * len(METHODS)
actual_rows = len(master_df)
print('Expected rows:', expected_rows)
print('Actual rows:', actual_rows)
assert actual_rows == expected_rows, f'Row mismatch: expected {expected_rows}, got {actual_rows}'
assert master_df['run_dir'].astype(str).str.len().gt(0).all(), 'Empty run_dir found'
assert master_df['best_checkpoint'].astype(str).str.len().gt(0).all(), 'Empty best_checkpoint found'
print('Basic output checks passed.')

In [ ]:
from IPython.display import Image, display
for p in benchmark['plot_paths']:
    print(p)
    display(Image(filename=p))